# Week 56, Day 1: The pattern map

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w056d1.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
import math

for n, label in ((20, "n = 20"), (500, "n = 500"), (5000, "n = 5,000"), (10 ** 5, "n = 100,000"), (10 ** 9, "n = 10^9")):
    costs = {"n!": math.factorial(min(n, 25)) if n <= 25 else float("inf"), "2^n": 2 ** n if n <= 60 else float("inf"),
             "n^3": n ** 3, "n^2": n ** 2, "n log n": round(n * math.log2(n)), "n": n}
    fine = [name for name, c in costs.items() if c <= 10 ** 7]
    print(label, fine)

In [ ]:
def ship_within(weights, days):
    """The least ship capacity that carries every parcel, in order, within the days."""
    def fits(capacity):
        used, load = 1, 0
        for w in weights:
            if load + w > capacity:
                used, load = used + 1, 0
            load += w
        return used <= days

    lo, hi = max(weights), sum(weights)
    while lo < hi:
        mid = (lo + hi) // 2
        if fits(mid):
            hi = mid
        else:
            lo = mid + 1
    return lo

print(ship_within([1, 2, 3, 4, 5, 6, 7, 8, 9, 10], 5), ship_within([3, 2, 2, 4, 1, 4], 3))

> **Key idea.** Input sizes set the budget: about 10 means n!, 20 means 2ⁿ, a few hundred n³, a few thousand n², 10⁵ to 10⁶ n log n or n, and 10⁹ or more needs log n or maths. Read the problem's shape and map it to a family: arrays to two pointers, windows and prefix sums; ordering to stacks, queues and heaps; connections to graphs; overlapping choices to DP; small sets to bitmasks; checkable answers to binary search on the answer. Look through disguises by asking what the states and moves are. When nothing fits, brute-force, find the repeated work, sort, and shrink the problem. Name the pattern and its cost before coding.

### Exercise: Read the budget

Write `affordable(n)`: the fastest-growing of `"n!"`, `"2^n"`, `"n^3"`, `"n^2"`, `"n log n"` and `"n"` whose cost for that n is at most 10⁷ steps (use `math.factorial`, `2 ** n`, `n ** 3`, `n ** 2`, `n * math.log2(n)` and `n`; for n = 1, `n log n` is 0). Check them in that order and return the first that fits.

In [ ]:
import math

def affordable(n):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_budget():\n    \"budgets\"\n    assert affordable(10) == \"n!\" and affordable(20) == \"2^n\" and affordable(200) == \"n^3\"\n    assert affordable(3000) == \"n^2\" and affordable(10 ** 5) == \"n log n\" and affordable(10 ** 7) == \"n\"\n    assert affordable(11) == \"2^n\" and affordable(215) == \"n^3\" and affordable(216) == \"n^2\" and affordable(1) == \"n!\"\n    assert affordable(10 ** 9) is None")

### Exercise: Pattern: closest to a target

Given a list of whole numbers (up to 10⁵ of them) and a target, write `closest_pair_sum(nums, target)`: the sum of two different positions' values that is closest to the target (the smaller sum on a tie). Choose a pattern that's O(n log n).

In [ ]:
def closest_pair_sum(nums, target):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random, time\ndef test_pairs():\n    \"sort and two pointers\"\n    assert closest_pair_sum([1, 4, 7, 10], 12) == 11 and closest_pair_sum([5, 5], 3) == 10\n    assert closest_pair_sum([1, 3], 3) == 4 and closest_pair_sum([-1, 2, 1, -4], 1) == 1 and closest_pair_sum([0, 2], 1) == 2\n    rng = random.Random(1)\n    for _ in range(200):\n        nums = [rng.randint(-20, 20) for _ in range(rng.randint(2, 9))]\n        t = rng.randint(-40, 40)\n        sums = [nums[i] + nums[j] for i in range(len(nums)) for j in range(i + 1, len(nums))]\n        assert closest_pair_sum(nums, t) == min(sums, key=lambda s: (abs(s - t), s))\n    big = [rng.randint(-10 ** 9, 10 ** 9) for _ in range(100_000)]\n    t0 = time.perf_counter()\n    closest_pair_sum(big, 12345)\n    assert time.perf_counter() - t0 < 3")

### Exercise: Pattern: how long since bigger?

For each day's temperature, write `days_since_warmer(temps)`: how many days back the most recent **strictly warmer** day was, or the day's index plus one if there was none. Up to 10⁵ days, so not O(n²).

In [ ]:
def days_since_warmer(temps):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\ndef test_spans():\n    \"monotonic stack\"\n    assert days_since_warmer([30, 20, 25, 40, 35]) == [1, 1, 2, 4, 1]\n    assert days_since_warmer([]) == [] and days_since_warmer([5, 5, 5]) == [1, 2, 3]\n    rng = random.Random(2)\n    for _ in range(200):\n        t = [rng.randint(0, 9) for _ in range(rng.randint(0, 12))]\n        want = []\n        for i, x in enumerate(t):\n            j = i - 1\n            while j >= 0 and t[j] <= x:\n                j -= 1\n            want.append(i - j)\n        assert days_since_warmer(t) == want\n    assert len(days_since_warmer(list(range(100_000)))) == 100_000")

### Exercise: Pattern: groups that merge

Friendships arrive one at a time as pairs of people numbered 0 to n − 1. Write `groups_after_each(n, pairs)`: a list of the number of separate friendship groups after each new pair. Up to 10⁵ people and pairs.

In [ ]:
def groups_after_each(n, pairs):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\ndef test_groups():\n    \"union-find\"\n    assert groups_after_each(4, [(0, 1), (2, 3), (1, 0), (1, 2)]) == [3, 2, 2, 1] and groups_after_each(3, []) == []\n    rng = random.Random(3)\n    for _ in range(100):\n        n = rng.randint(1, 8)\n        pairs = [(rng.randrange(n), rng.randrange(n)) for _ in range(rng.randint(0, 10))]\n        want, adj = [], {i: set() for i in range(n)}\n        for a, b in pairs:\n            adj[a].add(b)\n            adj[b].add(a)\n            seen, groups = set(), 0\n            for s in range(n):\n                if s not in seen:\n                    groups += 1\n                    stack = [s]\n                    seen.add(s)\n                    while stack:\n                        u = stack.pop()\n                        for v in adj[u]:\n                            if v not in seen:\n                                seen.add(v)\n                                stack.append(v)\n            want.append(groups)\n        assert groups_after_each(n, pairs) == want\n    out = groups_after_each(100_000, [(i, i + 1) for i in range(99_999)])\n    assert out[-1] == 1")

### Exercise: Pattern: the smallest largest part

Split a list of positive numbers into exactly k contiguous, non-empty parts so the **largest** part sum is as small as possible. Write `smallest_largest(nums, k)`. Up to 10⁵ numbers, each up to 10⁹. (A DP over split points would be O(n² × k): find the faster disguise.)

In [ ]:
def smallest_largest(nums, k):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import itertools, random, time\ndef brute(nums, k):\n    n = len(nums)\n    best = None\n    for cuts in itertools.combinations(range(1, n), k - 1):\n        b = (0,) + cuts + (n,)\n        worst = max(sum(nums[x:y]) for x, y in zip(b, b[1:]))\n        best = worst if best is None else min(best, worst)\n    return best\ndef test_split():\n    \"binary search on the answer\"\n    assert smallest_largest([7, 2, 5, 10, 8], 2) == 18 and smallest_largest([1, 2, 3, 4, 5], 2) == 9 and smallest_largest([5], 1) == 5\n    rng = random.Random(4)\n    for _ in range(100):\n        nums = [rng.randint(1, 20) for _ in range(rng.randint(1, 8))]\n        k = rng.randint(1, len(nums))\n        assert smallest_largest(nums, k) == brute(nums, k), (nums, k)\n    big = [rng.randint(1, 10 ** 9) for _ in range(100_000)]\n    t0 = time.perf_counter()\n    smallest_largest(big, 37)\n    assert time.perf_counter() - t0 < 6")

### Bug hunt: A greedy that isn't

This takes the largest stamp that fits, again and again, to make up postage of 12 with stamps of 1, 6 and 9, and prints `4` (9 + 1 + 1 + 1). Two 6s make 2. Replace the greedy with a fewest-coins DP so it prints `2`.

In [ ]:
stamps, postage = [1, 6, 9], 12
count = 0
for s in sorted(stamps, reverse=True):
    while postage >= s:
        postage -= s
        count += 1
print(count)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"DP, not greed\"\n    expect_output(\"2\")")

### Bug hunt: A search that never moves

A binary search on the answer must move a bound **past** `mid` when the check fails. This sets `lo = mid`, which loops forever when `hi = lo + 1`; so the loop here is cut off after 100 steps, and it prints `14` instead of the least capacity, `15`. Use `lo = mid + 1` (and drop the step limit).

In [ ]:
weights, days = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10], 5
def fits(c):
    used, load = 1, 0
    for w in weights:
        if load + w > c:
            used, load = used + 1, 0
        load += w
    return used <= days
lo, hi, steps = max(weights), sum(weights), 0
while lo < hi and steps < 100:
    mid = (lo + hi) // 2
    if fits(mid):
        hi = mid
    else:
        lo = mid
    steps += 1
print(lo)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"move past mid\"\n    expect_output(\"15\")")

### Bug hunt: Depth instead of breadth

The fewest moves in an unweighted grid needs **breadth**-first search. This uses a stack (depth-first), which finds *a* route, not the shortest: it prints `7` for a cell that is 3 moves away. Use a queue so it prints `3`.

In [ ]:
grid = ["....", ".##.", "...."]
start, goal = (0, 0), (2, 1)
dist, todo = {start: 0}, [start]
while todo:
    r, c = todo.pop()
    for nr, nc in ((r + 1, c), (r - 1, c), (r, c + 1), (r, c - 1)):
        if 0 <= nr < 3 and 0 <= nc < 4 and grid[nr][nc] == "." and (nr, nc) not in dist:
            dist[(nr, nc)] = dist[(r, c)] + 1
            todo.append((nr, nc))
print(dist[goal])

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"BFS\"\n    expect_output(\"3\")")

### Puzzle: Binary search on the answer

Arrange the lines of a binary search for the smallest value that passes a monotonic check `ok`. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
p = ___   # the usual technique for "n ≤ 18, visit each city once": "bitmask DP", "Dijkstra" or "two pointers"
q = ___   # the usual technique for "the cheapest route, weights up to 100": "BFS", "Dijkstra" or "DP over subsets"
r = ___   # the affordable cost for n = 3000 in Python: "n^3", "n^2" or "2^n"
s = ___   # True or False: if a quick check works for x and every bigger x, you can binary-search the answer
t = ___   # the technique for "next greater element for every position": "heap", "monotonic stack" or "trie"

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == \"bitmask DP\" and q == \"Dijkstra\" and r == \"n^2\", \"p, q and r\"\n    assert s is True and t == \"monotonic stack\", \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: the right tool, three times

Three small problems that look alike but need different tools. Name the pattern for each in a comment, then write:

- `fewest_hops(words, start, end)`: the fewest one-letter changes from `start` to `end`, each intermediate word in the list (all words the same length), or `None`. (Up to 5,000 words.)
- `cheapest_hops(words, start, end, cost)`: the same, but changing a letter from a to b costs `cost(a, b)` (a function returning a non-negative number); return the least total cost, or `None`.
- `hop_ways(words, start, end)`: how many **shortest** change sequences there are, mod 10⁹ + 7.

Generate neighbours by trying each position with each letter `a` to `z` and looking the result up in a set, not by comparing every pair of words.

In [ ]:
import heapq
from collections import deque

def fewest_hops(words, start, end):
    ___

def cheapest_hops(words, start, end, cost):
    ___

def hop_ways(words, start, end):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random, time\ndef test_hops():\n    \"BFS, Dijkstra, counting BFS\"\n    words = [\"hot\", \"dot\", \"dog\", \"lot\", \"log\", \"cog\"]\n    assert fewest_hops(words, \"hit\", \"cog\") == 4 and fewest_hops(words, \"hit\", \"cat\") is None and fewest_hops(words, \"hot\", \"hot\") == 0\n    assert hop_ways(words, \"hit\", \"cog\") == 2 and hop_ways(words, \"hit\", \"xyz\") == 0\n    flat = lambda a, b: 1\n    assert cheapest_hops(words, \"hit\", \"cog\", flat) == 4\n    alpha = lambda a, b: abs(ord(a) - ord(b))\n    best = cheapest_hops(words, \"hit\", \"cog\", alpha)\n    assert best == min(\n        abs(ord(\"i\") - ord(\"o\")) + abs(ord(\"h\") - ord(\"d\")) + abs(ord(\"t\") - ord(\"g\")) + abs(ord(\"d\") - ord(\"c\")),\n        abs(ord(\"i\") - ord(\"o\")) + abs(ord(\"h\") - ord(\"l\")) + abs(ord(\"t\") - ord(\"g\")) + abs(ord(\"l\") - ord(\"c\")))\n    rng = random.Random(5)\n    vocab = sorted({\"\".join(rng.choice(\"abcd\") for _ in range(4)) for _ in range(150)})\n    for _ in range(20):\n        s, e = rng.choice(vocab), rng.choice(vocab)\n        f = fewest_hops(vocab, s, e)\n        c = cheapest_hops(vocab, s, e, flat)\n        assert f == c and (hop_ways(vocab, s, e) == 0) == (f is None)\n    big = sorted({\"\".join(rng.choice(\"abcdefghij\") for _ in range(5)) for _ in range(5000)})\n    t0 = time.perf_counter()\n    fewest_hops(big, big[0], big[-1])\n    hop_ways(big, big[0], big[-1])\n    assert time.perf_counter() - t0 < 6")

## How to use this week

Two days of timed sets in Week 49 covered Weeks 41 to 48. This week covers everything from Week 41 to Week 55, with graphs, greedy algorithms, backtracking, dynamic programming and bits and maths added. The plan:

1. **Today**: the map, and recognition drills.
2. **Tuesday**: timed set 3, with hints and model solutions to open **after** the timer.
3. **Wednesday and Thursday**: timed sets 4 and 5, no hints, solutions in Saturday's debrief.
4. **Friday**: a mock interview, where explaining matters as much as the code.
5. **Saturday**: the debrief and second chances. Then a week of review and the DSA exam (Week 57).

One habit for every problem this week: write a one-line comment **before** your code, `# pattern: ..., cost: ...`. If you can't fill it in, that's the moment to stop and think, not to start typing.

## Sizes tell you the cost

Python does roughly 10⁷ simple steps per second (C does 10⁸ to 10⁹). Problems state their input sizes, and those sizes say which running times can work in a second or two:

| Largest n | Affordable | Usually means |
|---|---|---|
| about 10 | O(n!) | try every order (Week 16), backtracking (Week 52) |
| about 20 | O(2ⁿ × n) | every subset, bitmask DP (Week 54 Day 4) |
| about 100 to 500 | O(n³) | interval DP, Floyd-Warshall, triple loops (Weeks 50, 54) |
| about 5,000 | O(n²) | pairs, two-dimensional DP (Weeks 53, 54) |
| about 10⁵ to 10⁶ | O(n log n) or O(n) | sorting, heaps, binary search, one pass, hashing (Weeks 41 to 48) |
| 10⁹ to 10¹⁸ | O(log n) or O(1) | binary search on the answer, fast powers, maths (Weeks 18, 55) |

Values matter too: a knapsack with capacity 10⁴ is fine (O(n × W)); with capacity 10¹² it needs another idea (Week 53 Day 5).

## The map

Read the problem for its **shape**. The same shape usually means the same family of techniques.

| The problem asks about | Reach for | Week |
|---|---|---|
| pairs or triples in an array; sorted input | two pointers | 41 |
| the best contiguous stretch; "at most k" | sliding window | 41 |
| sums of ranges; "how many subarrays sum to k" | prefix sums with a dict | 41, 44 |
| counting, grouping, "have I seen it" | dict, set, Counter | 44 |
| matching brackets, undo, nesting | stack | 43 |
| next greater, spans, histograms | monotonic stack | 43 |
| window maximums | monotonic deque | 43 |
| the k largest, a running median, scheduling by time | heap | 47 |
| prefixes of many words | trie | 48 |
| hierarchy, recursion on parts | tree traversal, tree DP | 45, 46, 54 |
| sorted data with inserts and range queries | balanced BST or bisect | 46 |
| connections, reachability, groups | BFS, DFS, union-find | 50, 51 |
| fewest steps, unweighted | BFS (with extra state if needed) | 50, 51 |
| cheapest route, weights ≥ 0 | Dijkstra | 51 |
| order with dependencies | topological sort, DAG DP | 50, 54 |
| cheapest way to connect everything | minimum spanning tree | 51 |
| choose the best now, provably safe | greedy with an exchange argument | 52 |
| intervals: schedule, merge, cover | sort by end or start, then greedy or sweep | 49, 52 |
| all arrangements, constraints, puzzles | backtracking with pruning | 52 |
| best or count over choices that overlap | dynamic programming | 53, 54 |
| two strings | LCS or edit-distance table | 54 |
| a small set, "each exactly once" | bitmask DP | 54 |
| "smallest x such that ..." with a yes/no test | binary search on the answer | 18 |
| huge numbers, remainders, "mod 10⁹ + 7" | modular arithmetic, fast powers | 55 |
| bits, XOR, flags | bit tricks | 55 |

## Disguises

Many problems hide their shape:

- **A graph without saying "graph"**: words that differ by one letter (word ladder), states of a puzzle, cells of a grid, currencies and rates. Ask: what are the **states**, and what are the **moves**?
- **A greedy that isn't**: if a small counterexample breaks the obvious rule (coins 1, 3, 4 and amount 6), it's DP. If an exchange argument works, it's greedy.
- **DP dressed as search**: "the number of ways", "the fewest", "is it possible", with choices that overlap. Ask yesterday's question from Week 53: if you stopped halfway, what would you need to know?
- **Binary search on the answer**: "the smallest capacity that ships everything in d days", "the largest minimum distance". If you can **check** a candidate answer quickly, and checks are monotonic (if x works, everything bigger works), binary-search x.
- **Sorting first**: many problems become easy once the input is sorted (intervals, two pointers, greedy). O(n log n) is almost always affordable.

## When nothing fits

If no pattern comes to mind:

1. **Brute force first**, on paper or in code: try everything. It gives correct answers to test against.
2. **Find the repeated work.** Is the same sub-question answered many times? (DP or memoisation.) Is something rescanned that could be kept? (A running total, a window, a heap.) Is a search repeated that could be a lookup? (A dict or set.)
3. **Sort**, and look again.
4. **Shrink the problem**: solve it for n = 1, 2, 3 by hand and look for the step from n − 1 to n.
5. **Change what you compute**: binary-search the answer, count the complement, swap the roles (Week 54 Day 5's egg drop).

## Practice

These problems are short on purpose: the skill is recognising the pattern. Write your `# pattern:` comment first.

## Quiz

## Daily challenge

## Before you go

Look back at the last three problems you solved in any week, and write the `# pattern: ..., cost: ...` line each should have started with. Tomorrow: timed set 3.